In [1]:
# Bibliotecas para analise exploratoria de dados
from pathlib import Path

import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xml.etree.ElementTree as ET
import zipfile
import unicodedata
import re



In [7]:
# Fun??es utilit?rias

def col_index(cell_ref):
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    n = 0
    for ch in letters:
        n = n * 26 + ord(ch.upper()) - 64
    return n - 1

def quote_identifier(name):
    return "\"" + str(name).replace("\"", "\"\"") + "\""

def read_xlsx_first_sheet(path):
    with zipfile.ZipFile(path) as z:
        shared = []
        if "xl/sharedStrings.xml" in z.namelist():
            root = ET.fromstring(z.read("xl/sharedStrings.xml"))
            for si in root.findall("a:si", ns):
                parts = [t.text or "" for t in si.findall(".//a:t", ns)]
                shared.append("".join(parts))

        workbook = ET.fromstring(z.read("xl/workbook.xml"))
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        rid_to_target = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels.findall("rel:Relationship", rel_ns)}
        first_sheet = workbook.find("a:sheets", ns).findall("a:sheet", ns)[0]
        rid = first_sheet.attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]
        sheet_path = "xl/" + rid_to_target[rid].lstrip("/")
        sheet = ET.fromstring(z.read(sheet_path))

        rows = []
        max_col = 0
        for row in sheet.findall(".//a:sheetData/a:row", ns):
            values = []
            for cell in row.findall("a:c", ns):
                idx = col_index(cell.attrib.get("r", "A1"))
                while len(values) <= idx:
                    values.append(None)
                cell_type = cell.attrib.get("t")
                value_node = cell.find("a:v", ns)
                value = None if value_node is None else value_node.text
                if cell_type == "s" and value is not None:
                    value = shared[int(value)]
                elif cell_type == "inlineStr":
                    value = "".join(t.text or "" for t in cell.findall(".//a:t", ns))
                values[idx] = value
            max_col = max(max_col, len(values))
            rows.append(values)

    if not rows:
        return [], []
    rows = [row + [None] * (max_col - len(row)) for row in rows]
    header = [str(col).strip() if col is not None else f"coluna_{i + 1}" for i, col in enumerate(rows[0])]
    return header, rows[1:]

def normalize_column_name(name):
    normalized = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode("ascii")
    normalized = normalized.lower()
    normalized = re.sub(r"[^a-z0-9]+", "_", normalized).strip("_")
    return normalized or "coluna"

def unique_column_names(columns):
    seen = {}
    result = []
    for column in columns:
        base = normalize_column_name(column)
        count = seen.get(base, 0) + 1
        seen[base] = count
        result.append(base if count == 1 else f"{base}_{count}")
    return result

def can_parse_int(value):
    try:
        return float(str(value).replace(",", ".")).is_integer()
    except (TypeError, ValueError):
        return False

def can_parse_float(value):
    try:
        float(str(value).replace(",", "."))
        return True
    except (TypeError, ValueError):
        return False

def infer_sqlite_types(columns, rows, text_force=None):
    text_force = set(text_force or [])
    sqlite_types = {}
    for idx, column in enumerate(columns):
        values = [row[idx] for row in rows if idx < len(row) and row[idx] not in (None, "")]
        if column in text_force or not values:
            sqlite_types[column] = "TEXT"
        elif all(can_parse_int(value) for value in values):
            sqlite_types[column] = "INTEGER"
        elif all(can_parse_float(value) for value in values):
            sqlite_types[column] = "REAL"
        else:
            sqlite_types[column] = "TEXT"
    return sqlite_types

def coerce_value(value, sqlite_type):
    if value is None or value == "":
        return None
    if sqlite_type == "INTEGER":
        try:
            return int(float(str(value).replace(",", ".")))
        except ValueError:
            return value
    if sqlite_type == "REAL":
        try:
            return float(str(value).replace(",", "."))
        except ValueError:
            return value
    return str(value)

def load_data_from_db(db_path, query=None):
    query = query 
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(query, conn)

def create_table_from_rows(conn, table_name, headers, raw_rows, text_force=None):
    columns = unique_column_names(headers)
    sqlite_types = infer_sqlite_types(columns, raw_rows, text_force=text_force)
    rows = [tuple(coerce_value(value, sqlite_types[column]) for column, value in zip(columns, row)) for row in raw_rows]

    cursor = conn.cursor()
    cursor.execute(f"DROP TABLE IF EXISTS {quote_identifier(table_name)}")
    column_sql = ",\n    ".join(f"{quote_identifier(column)} {sqlite_types[column]}" for column in columns)
    cursor.execute(f"CREATE TABLE {quote_identifier(table_name)} (\n    {column_sql}\n)")

    if rows:
        placeholders = ", ".join("?" for _ in columns)
        quoted_columns = ", ".join(quote_identifier(column) for column in columns)
        cursor.executemany(
            f"INSERT INTO {quote_identifier(table_name)} ({quoted_columns}) VALUES ({placeholders})",
            rows,
        )

    return {
        "tabela": table_name,
        "linhas": len(rows),
        "colunas": len(columns),
        "colunas_criadas": ", ".join(columns),
    }

def create_sqlite_table_from_xlsx(conn, file_path, table_name=None, text_force=None):
    file_path = Path(file_path)
    table_name = table_name or normalize_column_name(file_path.stem)
    headers, raw_rows = read_xlsx_first_sheet(file_path)
    if not headers:
        raise ValueError(f"Arquivo sem cabe?alho/dados: {file_path}")
    summary = create_table_from_rows(conn, table_name, headers, raw_rows, text_force=text_force)
    summary["arquivo"] = file_path.name
    return summary

def create_sqlite_tables_from_folder(db_path, folder_path, text_force=None):
    folder_path = Path(folder_path)
    files = sorted(path for path in folder_path.glob("*.xlsx") if not path.name.startswith("~$"))
    summaries = []

    if not files:
        raise FileNotFoundError(f"Nenhum arquivo .xlsx encontrado em: {folder_path}")

    with sqlite3.connect(db_path) as conn:
        for file_path in files:
            summaries.append(create_sqlite_table_from_xlsx(conn, file_path, text_force=text_force))
        conn.commit()

    return pd.DataFrame(summaries)[["arquivo", "tabela", "linhas", "colunas", "colunas_criadas"]]

def create_sqlite_tables_from_folder_safe(db_path, folder_path, text_force=None):
    import shutil
    import tempfile
    import time
    from datetime import datetime

    db_path = Path(db_path).resolve()
    journal_path = db_path.with_name(db_path.name + "-journal")
    tmpdir = Path(tempfile.mkdtemp(prefix="ida_logcomex_work_"))
    tmp_db = tmpdir / db_path.name
    tmp_journal = tmpdir / journal_path.name

    shutil.copy2(db_path, tmp_db)
    if journal_path.exists():
        shutil.copy2(journal_path, tmp_journal)

    # Abre o banco tempor?rio uma vez para o SQLite recuperar/fechar journal pendente, se existir.
    with sqlite3.connect(tmp_db) as conn:
        integrity_before = conn.execute("PRAGMA integrity_check").fetchone()[0]
        if integrity_before != "ok":
            raise sqlite3.DatabaseError(f"Banco tempor?rio inv?lido antes da carga: {integrity_before}")
    if tmp_journal.exists():
        tmp_journal.unlink()

    summary = create_sqlite_tables_from_folder(tmp_db, folder_path, text_force=text_force)

    with sqlite3.connect(tmp_db) as conn:
        integrity_after = conn.execute("PRAGMA integrity_check").fetchone()[0]
        if integrity_after != "ok":
            raise sqlite3.DatabaseError(f"Banco tempor?rio inv?lido depois da carga: {integrity_after}")

    backup_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_db = db_path.with_name(f"{db_path.stem}.backup_{backup_stamp}{db_path.suffix}")
    backup_journal = journal_path.with_name(f"{db_path.stem}.backup_{backup_stamp}{db_path.suffix}-journal")

    time.sleep(0.5)
    db_path.replace(backup_db)
    if journal_path.exists():
        journal_path.replace(backup_journal)
    shutil.copy2(tmp_db, db_path)

    summary["banco_atualizado"] = str(db_path)
    summary["backup_banco_anterior"] = str(backup_db)
    return summary

def list_sqlite_tables(db_path):
    query = """
    SELECT name AS tabela
    FROM sqlite_master
    WHERE type = 'table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
    """
    return load_data_from_db(db_path, "sqlite_master", query)


def create_sqlite_table_from_xlsx_safe(db_path, file_path, table_name, text_force=None):
    import shutil
    import tempfile
    import time
    from datetime import datetime

    db_path = Path(db_path).resolve()
    journal_path = db_path.with_name(db_path.name + "-journal")
    tmpdir = Path(tempfile.mkdtemp(prefix="ida_logcomex_single_"))
    tmp_db = tmpdir / db_path.name
    tmp_journal = tmpdir / journal_path.name

    shutil.copy2(db_path, tmp_db)
    if journal_path.exists():
        shutil.copy2(journal_path, tmp_journal)

    with sqlite3.connect(tmp_db) as conn:
        integrity_before = conn.execute("PRAGMA integrity_check").fetchone()[0]
        if integrity_before != "ok":
            raise sqlite3.DatabaseError(f"Banco tempor?rio inv?lido antes da carga: {integrity_before}")
    if tmp_journal.exists():
        tmp_journal.unlink()

    with sqlite3.connect(tmp_db) as conn:
        summary = create_sqlite_table_from_xlsx(conn, file_path, table_name=table_name, text_force=text_force)
        conn.commit()
        integrity_after = conn.execute("PRAGMA integrity_check").fetchone()[0]
        if integrity_after != "ok":
            raise sqlite3.DatabaseError(f"Banco tempor?rio inv?lido depois da carga: {integrity_after}")

    backup_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_db = db_path.with_name(f"{db_path.stem}.backup_{backup_stamp}{db_path.suffix}")
    backup_journal = journal_path.with_name(f"{db_path.stem}.backup_{backup_stamp}{db_path.suffix}-journal")

    time.sleep(0.5)
    db_path.replace(backup_db)
    if journal_path.exists():
        journal_path.replace(backup_journal)
    shutil.copy2(tmp_db, db_path)

    summary["banco_atualizado"] = str(db_path)
    summary["backup_banco_anterior"] = str(backup_db)
    return pd.DataFrame([summary])


In [3]:
# Conectar ao banco existente e criar tabelas da pasta De-Para/Output GPT
db_path = Path("ida_logcomex.db")
table_name = "logcomex_importacoes_teste_explor"
depara_output_dir = Path("..") / "De-Para" / "Output GPT"

ns = {
    "a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
    "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
}
rel_ns = {"rel": "http://schemas.openxmlformats.org/package/2006/relationships"}

if not db_path.exists():
    raise FileNotFoundError(f"Banco n?o encontrado: {db_path.resolve()}")
if not depara_output_dir.exists():
    raise FileNotFoundError(f"Pasta n?o encontrada: {depara_output_dir.resolve()}")

df_tabelas_depara_criadas = create_sqlite_tables_from_folder_safe(db_path, depara_output_dir)
df_tabelas_banco = list_sqlite_tables(db_path)

display(df_tabelas_depara_criadas)
display(df_tabelas_banco)


,arquivo,tabela,linhas,colunas,colunas_criadas,banco_atualizado,backup_banco_anterior
0,depara_exportador_player.xlsx,depara_exportador_player,254,3,"provavel_exportador, player, data_regra",C:\Users\amanda.oliveiracunha\OneDrive - Therm...,C:\Users\amanda.oliveiracunha\OneDrive - Therm...
1,depara_paises_exportador.xlsx,depara_paises_exportador,202,4,"pais_de_aquisicao, pais_de_origem, provavel_ex...",C:\Users\amanda.oliveiracunha\OneDrive - Therm...,C:\Users\amanda.oliveiracunha\OneDrive - Therm...
2,depara_palavrachave_exportador.xlsx,depara_palavrachave_exportador,26485,3,"palavra_chave, provavel_exportador, data_regra",C:\Users\amanda.oliveiracunha\OneDrive - Therm...,C:\Users\amanda.oliveiracunha\OneDrive - Therm...


,tabela
0,depara_exportador_player
1,depara_paises_exportador
2,depara_palavrachave_exportador
3,ext_logcomex
4,logcomex_importacoes_teste_explor


In [4]:
# Criar tabela ext_logcomex a partir do arquivo da pasta Input
db_path = globals().get("db_path", Path("ida_logcomex.db"))
ns = globals().get("ns", {
    "a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
    "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
})
rel_ns = globals().get("rel_ns", {"rel": "http://schemas.openxmlformats.org/package/2006/relationships"})

input_dir = Path("..") / "Input"
ext_table_name = "ext_logcomex"
input_files = sorted(path for path in input_dir.glob("*.xlsx") if not path.name.startswith("~$"))

if len(input_files) != 1:
    raise RuntimeError(f"Esperado 1 arquivo .xlsx em {input_dir}, encontrados {len(input_files)}")

input_file = input_files[0]
df_ext_logcomex_criada = create_sqlite_table_from_xlsx_safe(db_path, input_file, ext_table_name)
df_ext_logcomex = load_data_from_db(db_path, ext_table_name)

display(df_ext_logcomex_criada)
display(df_ext_logcomex.head())
df_ext_logcomex.shape


,tabela,linhas,colunas,colunas_criadas,arquivo,banco_atualizado,backup_banco_anterior
0,ext_logcomex,280,41,"ano_mes, descricao_produto, provavel_incoterm,...",821135798562365fd781a118beaf0c8f.xlsx,C:\Users\amanda.oliveiracunha\OneDrive - Therm...,C:\Users\amanda.oliveiracunha\OneDrive - Therm...


,ano_mes,descricao_produto,provavel_incoterm,modal,ncm,pais_de_aquisicao,pais_de_origem,palavras_chave_de_descricao_do_produto,peso_liquido,provavel_exportador,...,relevancia_por_quantidade_2,palavra_chave_3,relevancia_por_valor_fob_3,relevancia_por_quantidade_3,palavra_chave_4,relevancia_por_valor_fob_4,relevancia_por_quantidade_4,palavra_chave_5,relevancia_por_valor_fob_5,relevancia_por_quantidade_5
0,202601,"1A137E7 ', '316L ', 'BIOMOLECULAS ', 'COMTUBUL...",CIP,AEREA,90272012,ESTADOS UNIDOS,SUÉCIA,None,2794.0,Cytiva,...,NaN,None,NaN,NaN,None,NaN,NaN,None,None,None
1,202603,"BIOMOLECULAS ', 'BIONOVIS ', 'COMTUBULACOES ',...",CIP,AEREA,90272012,SUÉCIA,ESTADOS UNIDOS,None,1430.0,Cytiva,...,NaN,None,NaN,NaN,None,NaN,NaN,None,None,None
2,202603,INSTRUMENTOS E APARELHOS PARA ANÁLISES FÍSICAS...,FCA,AEREA,90272012,None,URUGUAI,None,537.6,None,...,NaN,None,NaN,NaN,None,NaN,NaN,None,None,None
3,202601,"ESPECTRÔMETRO ', 'IRMS ', 'IRMS DELTA V ', 'MA...",EXW,AEREA,90272012,ALEMANHA,ALEMANHA,None,456.0,None,...,NaN,None,NaN,NaN,None,NaN,NaN,None,None,None
4,202602,INSTRUMENTOS E APARELHOS PARA ANÁLISES FÍSICAS...,FCA,None,90272012,URUGUAI,JAPÃO,None,481.2,None,...,NaN,None,NaN,NaN,None,NaN,NaN,None,None,None


(280, 41)

In [43]:
# Criar DataFrame a partir da tabela SQLite
query = f"SELECT * FROM depara_exportador_player;"
teste = load_data_from_db(db_path, query)

display(teste.head())
teste.shape

,provavel_exportador,player,data_regra
0,AB APPLIED BIOSYSTEMS UNITED STATES,THERMO FISHER,46148
1,QIAGEN GMBH FINLAND,QIAGEN,46148
2,QIAGEN DISTRIBUTION CHINA,QIAGEN,46148
3,QIAGEN DISTRIBUTION FRANCE,QIAGEN,46148
4,QIAGEN DISTRIBUTION GERMANY,QIAGEN,46148


(254, 3)

In [50]:
query1 = """


WITH base AS (
    SELECT
        e.*,
        CASE
            WHEN trim(e.provavel_exportador) <> '' THEN e.provavel_exportador
            WHEN EXISTS (
                SELECT 1
                FROM depara_palavrachave_exportador AS dpk
                WHERE instr(lower(e.descricao_produto), lower(dpk.palavra_chave)) > 0
            ) THEN (
                SELECT dpk.provavel_exportador
                FROM depara_palavrachave_exportador AS dpk
                WHERE instr(lower(e.descricao_produto), lower(dpk.palavra_chave)) > 0
                LIMIT 1
            )
            ELSE (
                SELECT dpe.provavel_exportador
                FROM depara_paises_exportador AS dpe
                WHERE lower(e.pais_de_origem) = lower(dpe.pais_de_origem)
                  AND lower(e.pais_de_aquisicao) = lower(dpe.pais_de_aquisicao)
                LIMIT 1
            )
        END AS exportador_mapeado,
        CASE
            WHEN trim(e.provavel_exportador) <> '' THEN 'registro_existente'
            WHEN EXISTS (
                SELECT 1
                FROM depara_palavrachave_exportador AS dpk
                WHERE instr(lower(e.descricao_produto), lower(dpk.palavra_chave)) > 0
            ) THEN 'palavra_chave'
            WHEN EXISTS (
                SELECT 1
                FROM depara_paises_exportador AS dpe
                WHERE lower(e.pais_de_origem) = lower(dpe.pais_de_origem)
                  AND lower(e.pais_de_aquisicao) = lower(dpe.pais_de_aquisicao)
            ) THEN 'pais'
            ELSE 'nenhuma'
        END AS regra_utilizada
    FROM ext_logcomex AS e
)

SELECT
    b.*,
    p.player
FROM base AS b
LEFT JOIN depara_exportador_player AS p
    ON lower(b.exportador_mapeado) = lower(p.provavel_exportador)


"""

In [51]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

teste1 = load_data_from_db(db_path, query1)

display(teste1.head(30))
teste1.shape

,ano_mes,descricao_produto,provavel_incoterm,modal,ncm,pais_de_aquisicao,pais_de_origem,palavras_chave_de_descricao_do_produto,peso_liquido,provavel_exportador,provavel_importador,relevancia_por_palavra_chave,urf_de_entrada,unidade_de_medida_comercializada_estimada,valor_cif_total,valor_cif_unitario,valor_fob_estimado_total,valor_fob_estimado_unitario,valor_frete_unitario,valor_seguro_unitario,cod_cgce,cod_cnae_importador,valor_seguro_total,qtd_de_operacoes_estimada,qtd_estatistica,unidade_de_medida_estatistica,palavra_chave_1,relevancia_por_valor_fob_1,relevancia_por_quantidade_1,palavra_chave_2,relevancia_por_valor_fob_2,relevancia_por_quantidade_2,palavra_chave_3,relevancia_por_valor_fob_3,relevancia_por_quantidade_3,palavra_chave_4,relevancia_por_valor_fob_4,relevancia_por_quantidade_4,palavra_chave_5,relevancia_por_valor_fob_5,relevancia_por_quantidade_5,exportador_mapeado,regra_utilizada,player
0,202601,"1A137E7 ', '316L ', 'BIOMOLECULAS ', 'COMTUBULACOES ', 'CROMATOGRAFIALIQUIDA ', 'CYTIVA ', 'ESCALA ', 'POLEGADA ', 'PURIFICACAO' INSTRUMENTOS E APARELHOS PARA ANÁLISES FÍSICAS OU QUÍMICAS (POR EXEMPLO: POLARÍMETROS, REFRACTÓMETROS, ESPECTRÓMETROS, ANALISADORES DE GASES OU DE FUMOS); INSTRUMENTOS E APARELHOS PARA ENSAIOS DE VISCOSIDADE, POROSIDADE, DILATAÇÃO, TENSÃO SUPERFICIAL OU S | BENS DE CAPITAL (EXCETO EQUIPAMENTOS DE TRANSPORTE) | PRODUTOS MANUFATURADOS | FABRICAÇÃO DE EQUIPAMENTOS DE MEDIÇÃO, TESTE, NAVEGAÇÃO E CONTROLE | INSTRUMENTOS E APARELHOS DE MEDIDA, DE VERIFICACAO, ETC",CIP,AEREA,90272012,ESTADOS UNIDOS,SUÉCIA,None,2794.0,Cytiva,None,None,AEROPORTO INTERNACIONAL DE SAO PAULO/GUARULHOS,NÚMERO (UNIDADE),1.313278e+06,1.313278e+06,1.291714e+06,1.291714e+06,22935.10,503.10,110,21.21-1/01,503.10,1,1,NÚMERO (UNIDADE),None,NaN,NaN,None,NaN,NaN,None,NaN,NaN,None,NaN,NaN,None,None,None,Cytiva,registro_existente,None
1,202603,"BIOMOLECULAS ', 'BIONOVIS ', 'COMTUBULACOES ', '485EA7 ', 'CROMATOGRAFIALIQUIDA ', 'CYTIVA ', 'EQUIPAMENTO ', 'ESCALA ', 'PURIFICACAO' INSTRUMENTOS E APARELHOS PARA ANÁLISES FÍSICAS OU QUÍMICAS (POR EXEMPLO: POLARÍMETROS, REFRACTÓMETROS, ESPECTRÓMETROS, ANALISADORES DE GASES OU DE FUMOS); INSTRUMENTOS E APARELHOS PARA ENSAIOS DE VISCOSIDADE, POROSIDADE, DILATAÇÃO, TENSÃO SUPERFICIAL OU S | BENS DE CAPITAL (EXCETO EQUIPAMENTOS DE TRANSPORTE) | PRODUTOS MANUFATURADOS | FABRICAÇÃO DE EQUIPAMENTOS DE MEDIÇÃO, TESTE, NAVEGAÇÃO E CONTROLE | INSTRUMENTOS E APARELHOS DE MEDIDA, DE VERIFICACAO, ETC",CIP,AEREA,90272012,SUÉCIA,ESTADOS UNIDOS,None,1430.0,Cytiva,None,None,AEROPORTO INTERNACIONAL DE SAO PAULO/GUARULHOS,NÚMERO (UNIDADE),9.069340e+05,9.069340e+05,8.966295e+05,8.966295e+05,10853.70,346.90,110,21.21-1/01,346.90,1,1,NÚMERO (UNIDADE),None,NaN,NaN,None,NaN,NaN,None,NaN,NaN,None,NaN,NaN,None,None,None,Cytiva,registro_existente,None
2,202603,"INSTRUMENTOS E APARELHOS PARA ANÁLISES FÍSICAS OU QUÍMICAS (POR EXEMPLO: POLARÍMETROS, REFRACTÓMETROS, ESPECTRÓMETROS, ANALISADORES DE GASES OU DE FUMOS); INSTRUMENTOS E APARELHOS PARA ENSAIOS DE VISCOSIDADE, POROSIDADE, DILATAÇÃO, TENSÃO SUPERFICIAL OU S | BENS DE CAPITAL (EXCETO EQUIPAMENTOS DE TRANSPORTE) | PRODUTOS MANUFATURADOS | FABRICAÇÃO DE EQUIPAMENTOS DE MEDIÇÃO, TESTE, NAVEGAÇÃO E CONTROLE | INSTRUMENTOS E APARELHOS DE MEDIDA, DE VERIFICACAO, ETC",FCA,AEREA,90272012,None,URUGUAI,None,537.6,None,None,None,ALF BELO HORIZONTE,NAO IDENTIFICADO,4.352333e+05,4.352333e+05,4.235816e+05,4.235816e+05,10251.20,2413.70,110,85.50-3/02,2413.70,1,1,NÚMERO (UNIDADE),None,NaN,NaN,None,NaN,NaN,None,NaN,NaN,None,NaN,NaN,None,None,None,ABBOTT LABORATORIES UNITED STATES,palavra_chave,ABBOTT
3,202601,"ESPECTRÔMETRO ', 'IRMS ', 'IRMS DELTA V ', 'MASSAS ', '1906F07 ', 'RAZÃO ', 'THERMO' INSTRUMENTOS E APARELHOS PARA ANÁLISES FÍSICAS OU QUÍMICAS (POR EXEMPLO: POLARÍMETROS, REFRACTÓMETROS, ESPECTRÓMETROS, ANALISADORES DE GASES OU DE FUMOS); INSTRUMENTOS E APARELHOS PARA ENSAIOS DE VISCOSIDADE, POROSIDADE, DILATAÇÃO, TENSÃO SUPERFICIAL OU S | BENS DE CAPITAL (EXCETO EQUIPAMENTO

(280, 44)